# Lesson 2 | How does digital hardware store 0.22?

In the previous lesson we built our first runnable **Leaky Integrate-and-Fire (LIF)** neuron. Its rule contains numbers such as `0.9`, `0.22`, and `1.0`.

In Python, we can use those numbers without thinking much about how they are stored. But before moving toward real digital hardware, a new question appears:

> **How does a digital system with finite hardware resources represent numbers with fractional parts?**

The primary new concept in this lesson is **finite-width numerical representation**.

## 1. Correct a common intuition: Python float is not an infinite-precision real number

In Lesson 1 we deliberately ignored number representation. Now it matters.

Python's common `float` type is typically a 64-bit binary floating-point format. It is flexible, but still finite precision and cannot represent every real number exactly. Decimal `0.1`, for example, is usually stored as a very close binary approximation.

So this lesson is not “perfect real numbers versus imperfect hardware.” More accurately, we are comparing two finite-precision representation styles:

- **floating-point number**: the position of the binary point can move through an exponent, giving a large dynamic range;
- **fixed-point number**: the binary point stays at a fixed position, making the rules simpler and hardware cost easier to control.

An FPGA can perform floating-point arithmetic too, but many accelerators carefully ask whether it is necessary because small fixed-width fixed-point arithmetic can be cheaper and easier to parallelize.

## 2. What is a bit, and why does width matter?

A **binary digit (bit)** is the basic unit of digital information, usually `0` or `1`.

Combining bits creates more possible encodings:

- 1 bit: 2 states;
- 2 bits: 4 states;
- 8 bits: 256 different encodings.

The number of bits used to store a value is its **bit width**.

More bits usually allow a larger range or finer precision, but consume more storage and arithmetic resources.

The hardware intuition for this lesson is:

> **Numeric format is not background bookkeeping; it is part of architecture.**

## 3. Fixed-point intuition: a ruler with fixed tick marks

Imagine a ruler whose smallest step is `1/16 = 0.0625`.

Then:

- `0.0` is representable;
- `0.0625` is representable;
- `0.125` is representable;
- `0.1875` is representable;
- but `0.10` does not land exactly on a tick and must be mapped to a nearby one.

Three terms appear here:

- **quantization**: mapping the original value onto a finite set of representable values;
- **rounding**: choosing which representable value to use when the original falls between two;
- **precision**: how fine the representable steps are.

Fixed point does not mean “no fractions.” It means the position of the binary point is fixed in advance.

## 4. What are `total_bits` and `frac_bits`?

For now we avoid ambiguous `Qm.n` naming conventions and record two explicit parameters:

- `total_bits`: total width;
- `frac_bits`: how many bits are devoted to fractional precision.

If `frac_bits = 4`, the scale is:

`scale = 2^4 = 16`

To encode a real value, multiply by 16 and store an integer. For example:

`0.25 × 16 = 4`

So integer code `4` represents the value `0.25`.

If the value does not land exactly on an available code, rounding is required.

In [ ]:
def quantize(x, total_bits=8, frac_bits=4):
    scale = 1 << frac_bits

    # signed integer range for this width
    min_i = -(1 << (total_bits - 1))
    max_i = (1 << (total_bits - 1)) - 1

    integer_code = round(x * scale)
    integer_code = max(min_i, min(max_i, integer_code))

    represented_value = integer_code / scale
    return integer_code, represented_value

for x in [0.1, 0.22, 0.9, 1.7, -0.3]:
    code, value = quantize(x, total_bits=8, frac_bits=4)
    print(f'{x:>5} -> integer code {code:>4} -> represented value {value}')

## 5. Observe: what did quantization change?

Look first at `0.1` and `0.22`.

With 4 fractional bits, they will usually become nearby representable values rather than remain exact.

Answer:

1. What is the smallest step for this format?
2. What value represents `0.22`, and what is the error?
3. If `frac_bits` increases, do the steps become coarser or finer?
4. If total width stays fixed while fractional bits increase, what happens to the largest representable magnitude?

We have reached a classic engineering tradeoff: **range and precision compete for finite bits.**

## 6. What is overflow, and why must saturation be specified?

When a result exceeds the range representable by the chosen width, **overflow** occurs.

There is no single unavoidable response to overflow. Common choices include:

- **saturation**: clamp above-range values to the maximum and below-range values to the minimum;
- **wraparound**: keep only the finite bits, potentially making a large positive result wrap into a negative one.

For membrane or synaptic accumulation, these policies can produce completely different neural activity.

Therefore a numeric specification cannot merely say “use 8 bits.” It must also define rounding and overflow policy.

In [ ]:
def signed_limits(total_bits):
    return -(1 << (total_bits - 1)), (1 << (total_bits - 1)) - 1

for bits in [4, 8, 12]:
    lo, hi = signed_limits(bits)
    print(f'{bits:2d} signed bits -> integer code range [{lo}, {hi}]')

## 7. Put finite width back into the LIF neuron

Now we make the previous LIF obey the same fixed-point rules at each update.

This is not the final hardware implementation. It is a Python **numeric reference model** for the rules future hardware should follow.

In [ ]:
def qvalue(x, total_bits, frac_bits):
    return quantize(x, total_bits, frac_bits)[1]

def run_lif_quantized(inputs, total_bits, frac_bits, alpha=0.9, threshold=1.0, reset=0.0):
    q = lambda x: qvalue(x, total_bits, frac_bits)

    v = q(0.0)
    spikes = []
    trace = []

    q_alpha = q(alpha)
    q_threshold = q(threshold)
    q_reset = q(reset)

    for t, current in enumerate(inputs):
        q_current = q(current)
        candidate_v = q(q_alpha * v + q_current)
        spike = candidate_v >= q_threshold
        v = q_reset if spike else candidate_v

        if spike:
            spikes.append(t)
        trace.append(v)

    return spikes, trace

inputs = [0.22] * 30

for fmt in [(8, 4), (12, 8), (16, 12)]:
    spikes, trace = run_lif_quantized(inputs, *fmt)
    print(f'total_bits={fmt[0]:2d}, frac_bits={fmt[1]:2d} -> spikes {spikes}')

## 8. Why can a small numeric error change spike timing?

If membrane state is far from threshold, a small quantization error may not matter.

Near threshold, however, `0.99` versus `1.01` can mean:

- one implementation spikes now;
- another spikes one step later;
- the two trajectories may then diverge.

So when we later compare Python, CPU, and FPGA implementations, we will not only ask whether average numeric error is small. We also care about whether the **spike sequence** remains consistent or within a defined tolerance.

## 9. Try It: change one design parameter at a time

Run three experiments, predicting first:

1. hold `total_bits=8`, compare `frac_bits=2, 4, 6`;
2. hold `frac_bits=4`, compare `total_bits=6, 8, 12`;
3. increase inputs enough to approach overflow and observe saturation.

Record:

- representable range;
- smallest step;
- spike times;
- difference from the floating-point version in Lesson 1.

## 10. AI Task

Ask AI to help:

- sweep width combinations automatically;
- generate error tables;
- plot floating-point versus fixed-point membrane trajectories;
- search for a small width that preserves the spike sequence on a chosen test set.

Require every result to state `total_bits`, `frac_bits`, rounding rule, and overflow rule. Do not accept ambiguous labels such as simply “Q8.”

## 11. Human Check

Without AI, explain:

- what a bit is and why width is a finite resource;
- the core difference between floating point and fixed point;
- the difference between quantization and rounding;
- why overflow requires an explicit rule;
- why saturation and wraparound can generate very different network behavior;
- why two numerically close trajectories can still have different spike timing.

## 12. Engineering Handoff

The mature formal reference will live in:

`python/reference/lif_fixed.py`

The selected format, rounding rule, and overflow policy will be written into MDD/TDD as a contract for future hardware.

**Register-Transfer Level (RTL)** is a hardware-design level we will use later to describe stored state and data movement across clock cycles. For today, recognizing the term is enough; writing RTL is not yet required.

## 13. Project Trace

- Lesson ID: `LSN-002`
- Engineering slice: `RMD-002`
- Related product design: `FR1 / FR2`
- Initial tests: `T-005 ~ T-006`

These are traceability IDs, not vocabulary to memorize.

## 14. Exit Ticket

Before continuing, you should be able to:

1. explain bit, floating point, fixed point, quantization, rounding, overflow, and saturation;
2. estimate the smallest step from `total_bits` and `frac_bits`;
3. explain the tradeoff between range and precision;
4. show experimentally how width changes LIF spike timing;
5. understand that numeric format itself is a hardware-design choice.

The next lesson asks a subtler question:

> If two people both say they implemented LIF, but one uses `>=` and the other uses `>`, did they implement the same model?